# Integrated clinical and multi-trait polygenic modelling of complex polypharmacy

## Notes before running the scripts
- Remember to replace all data and working directory paths with your actual paths.
- Dependency: glmnet, pROC, Matrix, data.table, dplyr, ggplot2, glue, nestedcv, caret, foreach, doParallel

In [ ]:
library(glmnet)
library(pROC)
library(Matrix)
library(data.table)

library(dplyr)
library(ggplot2)
library(parallel)
library(glue)

library(nestedcv)
library(doParallel)
library(foreach)
library(caret)

In [ ]:
working_dir <- "/working_directory/polypharmacy_PGS" # set your working path

In [ ]:
df_file <- glue("{working_dir}/polypharmacy_all_ancestry_multi_PGS.csv") 

all_ancestry_df <- read.csv(df_file, stringsAsFactors = FALSE) # read data
head(all_ancestry_df)

In [ ]:
prs_cols <- c( 
 'T1_anx2026_b37_PGS_PRScs',
 'T1_bip2025_b3738_PGS_PRScs',
 'T1_ed2025_b3738_PGS_PRScs',
 'T1_id2022_b3738_PGS_PRScs',
 'T1_mdd2025_b37_PGS_PRScs',
 'T1_ocd2025_b37_PGS_PRScs',
 'T1_ptsd2024_b37_PGS_PRScs',
 'T1_scz2025_PGS_PRScs',
 'T1_sud2023_b37_PGS_PRScs',
 'T2_extraversion2015_b37_PGS_PRScs',
 'T2_loneliness2025_b3738_PGS_PRScs',
 'T2_mtag2025_PGS_PRScs',
 'T2_neuroticism2025_b3738_PGS_PRScs',
 'T2_rtb2023_b3738_PGS_PRScs',
 'T2_swb2024_b3738_PGS_PRScs',
 'T3_cad2025_multi_b3738_PGS_PRScs',
 'T3_chronic_pain_2024_multi_b3738_PGS_PRScs',
 'T3_insomnia2024_multi_b3738_PGS_PRScs',
 'T3_migraine2025_eur_B3738_PGS_PRScs',
 'T3_t2d2025_b3738_PGS_PRScs',
 'T4_bmi2025_b3738_PGS_PRScs',
 'T4_crp2025_multi_b3738_PGS_PRScs',
 'T5_adhd2025_b3738_PGS_PRScs',
 'T5_asd2019_b37_PGS_PRScs',
 'T5_intelligence2025_multi_PGS_PRScs',
 'T5_neuro2025_eur_PGS_PRScs',
 'T5_ts2019_b37_PGS_PRScs'
)

In [ ]:
df <- all_ancestry_df

df$Sex <- factor(
  df$Sex,
  levels = c("Male", "Female", "Other"),
  ordered = FALSE
)

df$education_level <- factor(
  df$education_level,
  levels = c(
    "Advanced degree",
    "College graduate",
    "College one to three",
    "Twelve or GED",
    "Nine through eleven",
    "One to eight or Never attended",
      "Other"
  ),
  ordered = FALSE
)

df$income_level <- factor(
  df$income_level,
  levels = c(
     ">200k",
      "150k-200k",
      "100k-150k",
      "75k-100k",
      "50k-75k",
      "35k-50k",
      "25k-35k",
      "10k-25k",
      "<10k",
      "Other"
  ),
  ordered = FALSE
)

df$marital_status <- factor(
  df$marital_status,
  levels = c(
    "Married",
    "Living with partner",
    "Separated",
    "Widowed",
    "Divorced",
    "Never married",
    "Other"
  ),
  ordered = FALSE
)

df$Age <- as.numeric(df$Age)

df$psychiatric_multimorbidity <- as.numeric(df$psychiatric_multimorbidity)
df$CCI <- as.numeric(df$CCI)

pc_cols  <- paste0("PC", 1:10)

scale_cols <- c("Age", 'psychiatric_multimorbidity', prs_cols, pc_cols)
df[scale_cols] <- scale(df[scale_cols])

vars <- c(
  "Sex", "education_level", "income_level", "marital_status", scale_cols
) # "race_group", 

df_sub <- df[, vars]

X <- fastDummies::dummy_cols(
  df_sub,
  select_columns = c("Sex", "education_level", "income_level", "marital_status"), #"race_group", 
  remove_first_dummy = FALSE,
  remove_selected_columns = TRUE
)
X <- as.matrix(X[, setdiff(names(X), "y")])

y <- df$concurrent_polypharmacy_binary
y <- factor(y, levels = c(0, 1), labels = c("control", "case"))

In [ ]:
colnames(X)

In [ ]:
head(X)

In [ ]:
set.seed(123, "L'Ecuyer-CMRG")

In [ ]:
smote_out <- smote(y, X)
smote_y <- smote_out$y
smote_X <- smote_out$x


In [ ]:
dim(smote_X)

In [ ]:
head(smote_X)

In [ ]:
tg <- expand.grid(alpha = seq(0, 1, length = 20),
                  lambda = 10^seq(-4, 1, length = 100))

# 2. Use 'family = "binomial"' and appropriate metrics
smote_elastic_ncv <- nestcv.train(y = smote_y, x = smote_X,
               method = "glmnet",
               family = "binomial",    # Force logistic regression
               metric = "ROC",         # Optimize for AUC
               savePredictions = "final",
               outer_method = "cv",
               n_outer_folds = 10,
               n_inner_folds = 10,
               trControl = trainControl(method = "cv", 
                                        number = 10,
                                        classProbs = TRUE, 
                                        summaryFunction = twoClassSummary),                              
               tuneGrid = tg, 
               cv.cores = 10)


smote_elastic_ncv$summary

In [ ]:
summary(smote_elastic_ncv)

In [ ]:
library(dplyr)

# -------------------------------------------------
# 1. Extract outer CV predictions with fold ID
# -------------------------------------------------
all_preds <- do.call(rbind, lapply(seq_along(smote_elastic_ncv$outer_result), function(i) {
  
  df <- smote_elastic_ncv$outer_result[[i]]$preds
  
  data.frame(
    fold = i,
    obs  = df$testy,
    prob = df$predyp
  )
}))

# -------------------------------------------------
# 2. CLEAN outcome variable (FIX FOR YOUR ERROR)
# -------------------------------------------------

# convert safely to character first
all_preds$obs <- as.character(all_preds$obs)

# map possible encodings to 0/1
all_preds$obs <- dplyr::recode(
  all_preds$obs,
  "case" = 1,
  "control" = 0
)

# convert to numeric
all_preds$obs <- as.numeric(all_preds$obs)

# -------------------------------------------------
# 3. Clean probabilities
# -------------------------------------------------
all_preds$prob <- as.numeric(all_preds$prob)

# avoid log(0) issues
eps <- 1e-15
all_preds$prob <- pmin(pmax(all_preds$prob, eps), 1 - eps)

# -------------------------------------------------
# 4. Compute per-fold McFadden pseudo-R²
# -------------------------------------------------
r2_by_fold <- all_preds %>%
  group_by(fold) %>%
  summarise(
    
    # model log-likelihood
    loglik_model = sum(
      obs * log(prob) + (1 - obs) * log(1 - prob)
    ),
    
    # null model probability
    p_null = mean(obs),
    
    # null log-likelihood
    loglik_null = sum(
      obs * log(p_null) + (1 - obs) * log(1 - p_null)
    ),
    
    # McFadden pseudo-R²
    R2 = 1 - (loglik_model / loglik_null),
    
    .groups = "drop"
  )

# -------------------------------------------------
# 5. Summary statistics
# -------------------------------------------------
mean_r2 <- mean(r2_by_fold$R2, na.rm = TRUE)
sd_r2   <- sd(r2_by_fold$R2, na.rm = TRUE)

# output
mean_r2
sd_r2
r2_by_fold

In [ ]:
library(boot)

# function to compute R2
r2_fun <- function(data, indices) {
  
  d <- data[indices, ]
  
  loglik_model <- sum(
    d$obs * log(d$prob) +
    (1 - d$obs) * log(1 - d$prob)
  )
  
  p_null <- mean(d$obs)
  
  loglik_null <- sum(
    d$obs * log(p_null) +
    (1 - d$obs) * log(1 - p_null)
  )
  
  R2 <- 1 - (loglik_model / loglik_null)
  
  return(R2)
}

set.seed(123)

boot_out <- boot(
  data = all_preds,
  statistic = r2_fun,
  R = 1000
)

# SE
se_r2 <- sd(boot_out$t)

# CI
ci_r2 <- quantile(boot_out$t, c(0.025, 0.975))

se_r2
ci_r2

In [ ]:
saveRDS(smote_elastic_ncv, file = glue("{working_dir}/nested_cv_elastic_net_smote_all_ancestry.rds"))

In [ ]:
smote_elastic_ncv <- readRDS(glue("{working_dir}/nested_cv_elastic_net_smote_all_ancestry.rds"))

In [ ]:
names(smote_elastic_ncv)

In [ ]:
all_preds <- do.call(rbind, lapply(smote_elastic_ncv$outer_result, function(fold) {
  data.frame(
    obs  = fold$preds$testy,
    prob = fold$preds$predyp
  )
}))

library(pROC)

roc_obj <- roc(response = all_preds$obs,
               predictor = all_preds$prob,
               levels = c("control", "case"))  # control = negative, case = positive

auc(roc_obj)

ci.auc(roc_obj)

In [ ]:
class(smote_elastic_ncv$final_fit)

In [ ]:
glmnet_model <- smote_elastic_ncv$final_fit$finalModel

In [ ]:
glmnet_model <- smote_elastic_ncv$final_fit$finalModel

coef_mat <- coef(glmnet_model, s = smote_elastic_ncv$finalTune$lambda)

final_coef <- as.matrix(coef_mat)

final_coef <- final_coef[rownames(final_coef) != "(Intercept)", 1]



In [ ]:
rename_dict <- c(
  "T1_anx2026_b37_PGS_PRScs"                  = "Anxiety disorder PGS",
  "T1_bip2025_b3738_PGS_PRScs"                = "Bipolar disorder PGS",
  "T1_ed2025_b3738_PGS_PRScs"                 = "Eating disorder PGS",
  "T1_id2022_b3738_PGS_PRScs"                 = "Internalizing disorder PGS",
  "T1_mdd2025_b37_PGS_PRScs"                  = "Major depressive disorder PGS",
  "T1_ocd2025_b37_PGS_PRScs"                  = "Obsessive-compulsive disorder PGS",
  "T1_ptsd2024_b37_PGS_PRScs"                 = "Post traumatic stress disorder PGS",
  "T1_scz2025_PGS_PRScs"                      = "Schizophrenia PGS" ,
  "T1_sud2023_b37_PGS_PRScs"                  = "Substance use disorder PGS",
                
  "T2_extraversion2015_b37_PGS_PRScs"         = "Extraversion PGS",
  "T2_loneliness2025_b3738_PGS_PRScs"         = "Loneliness PGS",
  "T2_mtag2025_PGS_PRScs"                     = "Educational attainment PGS",
  "T2_neuroticism2025_b3738_PGS_PRScs"        = "Neuroticism PGS",
  "T2_rtb2023_b3738_PGS_PRScs"                = "Risk taking PGS",
  "T2_swb2024_b3738_PGS_PRScs"                = "Subjective wellbeing PGS",
                
  "T3_cad2025_multi_b3738_PGS_PRScs"          = "Coronary artery disease PGS",
  "T3_chronic_pain_2024_multi_b3738_PGS_PRScs"= "Chronic pain PGS",
  "T3_insomnia2024_multi_b3738_PGS_PRScs"     = "Insomnia PGS",
  "T3_migraine2025_eur_B3738_PGS_PRScs"       = "Migraine PGS",
  "T3_t2d2025_b3738_PGS_PRScs"                = "Type 2 diabetes PGS",
  
  "T4_bmi2025_b3738_PGS_PRScs"                = "Body mass index PGS",
  "T4_crp2025_multi_b3738_PGS_PRScs"          = "C-reactive protein PGS",
                
  "T5_adhd2025_b3738_PGS_PRScs"               = "ADHD PGS",
  "T5_asd2019_b37_PGS_PRScs"                  = "Autism spectrum disorder PGS",
  "T5_intelligence2025_multi_PGS_PRScs"       = "Intelligence PGS",
  "T5_neuro2025_eur_PGS_PRScs"                = "Motor development PGS",
  "T5_ts2019_b37_PGS_PRScs"                   = "Tourette syndrome PGS",

  "Age"                                                   = "Age",
  "psychiatric_multimorbidity"                            = "No. of psychiatric diagnoses",
  
  # Sex
  "Sex_Male"                                              = "Sex (Male)",
  "Sex_Female"                                            = "Sex (Female)",
  "Sex_Other"                                             = "Sex (Other)",

  
  # Education Level
  "education_level_Advanced degree"                       = "Education (Advanced Degree)",
  "education_level_College graduate"                      = "Education (College Graduate)",
  "education_level_College one to three"                  = "Education (College one to three)",
  "education_level_Twelve or GED"                         = "Education (Twelve or GED)",
  "education_level_Nine through eleven"                   = "Education (Nine through eleven)",
  "education_level_One to eight or Never attended"        = "Education (One to eight or Never attended)",
  "education_level_Other"                                 = "Education (Other)",
  
  # Annual Income Level
  "income_level_>200k"                                    = "Annual income (> $200k)",
  "income_level_150k-200k"                                = "Annual income ($150k - $200k)",
  "income_level_100k-150k"                                = "Annual income ($100k - $150k)",
  "income_level_75k-100k"                                 = "Annual income ($75k - $100k)",
  "income_level_50k-75k"                                  = "Annual income ($50k - $75k)",
  "income_level_35k-50k"                                  = "Annual income ($35k - $50k)",
  "income_level_25k-35k"                                  = "Annual income ($25k - $35k)",
  "income_level_10k-25k"                                  = "Annual income ($10k - $25k)",
  "income_level_<10k"                                     = "Annual income (< $10k)",
  "income_level_Other"                                    = "Annual income (Other)",
  
  # Marital Status
  "marital_status_Married"                                = "Marital status (Married)",
  "marital_status_Living with partner"                    = "Marital status (Living with partner)",
  "marital_status_Separated"                              = "Marital status (Separated)",
  "marital_status_Widowed"                                = "Marital status (Widowed)",
  "marital_status_Divorced"                               = "Marital status (Divorced)",
  "marital_status_Never married"                          = "Marital status (Never married)",
  "marital_status_Other"                                  = "Marital status (Other)"
)

In [ ]:
df <- data.frame(
  feature = names(final_coef),
  coef = final_coef
)

df$tier <- ifelse(grepl("T1", df$feature), "Psychiatric diagnosis (Tier(T)1)",
           ifelse(grepl("T2", df$feature), "Behavioural and psychological trait (T2)",
           ifelse(grepl("T3", df$feature), "Physical health diagnosis (T3)",
           ifelse(grepl("T4", df$feature), "Physical health trait (T4)",
           ifelse(grepl("T5", df$feature), "Neurodevelopment and intelligence (T5)",
           ifelse(grepl("marital_status", df$feature), "Marital status", 
           ifelse(grepl("income_level", df$feature), "Annual income", 
           ifelse(grepl("education_level", df$feature), "Education level", 
           ifelse(grepl("race_group", df$feature), "Ethnicity", 
           ifelse(grepl("psychiatric_multimorbidity", df$feature), "No. of psychiatric diagnoses", 
           ifelse(grepl("CCI", df$feature), "CCI score", 
                  "Age,sex")))))))))))

df$feature <- ifelse(
  is.na(rename_dict[df$feature]),
  df$feature,
  rename_dict[df$feature]
)

df <- df[df$coef != 0, ]
df$abs_coef <- abs(df$coef)
df <- df[order(df$abs_coef), ]

colors <- c(
    "Psychiatric diagnosis (Tier(T)1)" = "#e31a1c",
    "Behavioural and psychological trait (T2)" = "#fb9a99",    
    "Physical health diagnosis (T3)" = "#ff7f00",
    "Physical health trait (T4)" = "#fdbf6f",
    "Neurodevelopment and intelligence (T5)" = "#33a02c",
    "Marital status" = "#a6cee3",
    "Annual income"  = "#cab2d6",
    "Education level" = "#6a3d9a",
    "No. of psychiatric diagnoses" = "#1f78b4",
    "Age,sex" = "grey"
)

df$color <- colors[df$tier]

df_no_pc <- df[!grepl("^PC", df$feature), ]
df_select <- df_no_pc[!grepl("Sexunknown", df_no_pc$feature), ]

In [ ]:
df$feature

In [ ]:
df_select$tier <- factor(df_select$tier, levels = c(
    "Psychiatric diagnosis (Tier(T)1)",
    "Behavioural and psychological trait (T2)",    
    "Physical health diagnosis (T3)",
    "Physical health trait (T4)",
    "Neurodevelopment and intelligence (T5)",
    "Marital status",
    "Annual income",
    "Education level",
    "No. of psychiatric diagnoses",
    "Age,sex"
))

In [ ]:
df_select$feature

In [ ]:
library(ggplot2)

plot_p = ggplot(df_select, aes(
  x = coef,
  y = reorder(feature, abs(coef)),
  fill = tier
)) +
  geom_bar(stat = "identity") +
  geom_vline(xintercept = 0) +
  scale_fill_manual(values = colors, breaks = levels(df_select$tier)) +
    guides(
      fill = guide_legend(
        nrow = 5,
        ncol = 2,
        byrow = TRUE,
        keywidth = 0.5,
        keyheight = 0.5
      )
    ) + 
  theme_bw(base_size = 6, base_family = "Arial") + 
  #theme_classic() +
  theme(
    legend.position = "bottom",
    legend.justification = "right",
    legend.box.just = "right",
    legend.margin = margin(t = 0, r = 0, l=0),
    legend.title = element_text(size = 5),
    legend.text = element_text(size = 5)  
  ) +
labs(y = "Sociodemographic, clinical or polygenic score", x = "Coefficient estimate", fill = "Conceptual\ntrait\ndomain")

In [ ]:
library(ggplot2)

ggsave(
  filename = glue("{working_dir}/Figure/multi_clinical_PGS_nested_cv_elastic_net_smote_coefficients.pdf"),
  plot = plot_p,
  width = 100,
  height = 130,
  units = "mm",
  device = cairo_pdf
)


In [ ]:
df_select_PGS <- df_select

In [ ]:
df_select_PGS$tier <- factor(df_select_PGS$tier, levels = c(
    "Psychiatric diagnosis (Tier(T)1)",
    "Behavioural and psychological trait (T2)",    
    "Physical health diagnosis (T3)",
    "Physical health trait (T4)",
    "Neurodevelopment and intelligence (T5)"
))

In [ ]:
df_select_PGS <- df_select[grepl("PGS$", df_select$feature), ]

In [ ]:
library(ggplot2)

plot_p3 = ggplot(df_select_PGS, aes(
  x = coef,
  y = reorder(feature, abs(coef)),
  fill = tier
)) +
  geom_bar(stat = "identity") +
  geom_vline(xintercept = 0) +
coord_cartesian(xlim = c(-0.1, 0.1)) +
  scale_fill_manual(values = colors, breaks = levels(df_select_PGS$tier)) +
    guides(
      fill = guide_legend(
        nrow = 3,
        ncol = 2,
        byrow = TRUE,
        keywidth = 0.5,
        keyheight = 0.5
      )
    ) + 
  theme_bw(base_size = 6, base_family = "Arial") + 
  theme(
    legend.position = "bottom",
    legend.justification = "right",
    legend.box.just = "right",
    legend.margin = margin(t = 0, r = 0, l=0),
    legend.title = element_text(size = 5),
    legend.text = element_text(size = 5)  
  ) +
labs(y = "Polygenic score (PGS)", x = "Coefficient estimate", fill = "Conceptual\ntrait\ndomain")

In [ ]:
# PDF (vector format; preferred for journals)
ggsave(
  filename = glue("{working_dir}/Figure/multi_PGS_plot_nested_cv_elastic_net_smote_coefficients.pdf"),
  plot = plot_p3,
  width = 70,
  height = 90,
  units = "mm",
  device = cairo_pdf
)